In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge


In [2]:
# Загрузка данных
train = pd.read_csv('/kaggle/input/home-data-for-ml-course/train.csv')
test = pd.read_csv('/kaggle/input/home-data-for-ml-course/test.csv')

In [3]:
# Сохраняем Id для сабмишена
test_ids = test['Id']

In [4]:
# Разделяем целевую переменную
y = train['SalePrice']
X = train.drop(['SalePrice', 'Id'], axis=1)
X_test = test.drop('Id', axis=1)

# Определяем типы признаков
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Для категориальных признаков: заполняем пропуски специальным значением
X[categorical_features] = X[categorical_features].fillna('Missing')
X_test[categorical_features] = X_test[categorical_features].fillna('Missing')


# Для числовых признаков: заполняем медианой
X[numeric_features] = X[numeric_features].fillna(X[numeric_features].median())
X_test[numeric_features] = X_test[numeric_features].fillna(X[numeric_features].median())

In [5]:
# Создаём трансформер для категориальных переменных (Ordinal Encoding)
categorical_transformer = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# Пайплайн для предобработки
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),  # стандартизация числовых
        ('cat', categorical_transformer, categorical_features)
    ])

# Логарифмируем целевую переменную (стабилизирует обучение)
y_log = np.log1p(y)  # log(1 + x)

# Разделяем обучающий набор на train/val
X_train, X_val, y_train_log, y_val_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

In [6]:
# -----------------------------
# 2. Модель: RandomForest
# -----------------------------
print("Обучаем RandomForest...")
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])
pipeline_rf.fit(X_train, y_train_log)
y_pred_rf_log = pipeline_rf.predict(X_val)
rmsle_rf = np.sqrt(mean_squared_log_error(np.expm1(y_val_log), np.expm1(y_pred_rf_log)))
print(f"RMSLE RandomForest: {rmsle_rf:.4f}")

Обучаем RandomForest...
RMSLE RandomForest: 0.1450


In [7]:
# Предсказание на тесте
predictions_rf_log = pipeline_rf.predict(X_test)
predictions_rf = np.expm1(predictions_rf_log)

In [8]:
# -----------------------------
# 3. Модель: XGBoost
# -----------------------------
print("Обучаем XGBoost...")
pipeline_xg = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='rmse'
    ))
])
pipeline_xg.fit(X_train, y_train_log)
y_pred_xg_log = pipeline_xg.predict(X_val)
rmsle_xg = np.sqrt(mean_squared_log_error(np.expm1(y_val_log), np.expm1(y_pred_xg_log)))
print(f"RMSLE XGBoost: {rmsle_xg:.4f}")

Обучаем XGBoost...
RMSLE XGBoost: 0.1370


In [9]:
# Предсказание на тесте
predictions_xg_log = pipeline_xg.predict(X_test)
predictions_xg = np.expm1(predictions_xg_log)

In [10]:
# -----------------------------
# 4. Модель: LightGBM
# -----------------------------
print("Обучаем LightGBM...")
pipeline_lg = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=10,
        random_state=42,
        n_jobs=-1
    ))
])
pipeline_lg.fit(X_train, y_train_log)
y_pred_lg_log = pipeline_lg.predict(X_val)
rmsle_lg = np.sqrt(mean_squared_log_error(np.expm1(y_val_log), np.expm1(y_pred_lg_log)))
print(f"RMSLE LightGBM: {rmsle_lg:.4f}")

Обучаем LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3146
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 74
[LightGBM] [Info] Start training from score 12.030658
RMSLE LightGBM: 0.1362


In [11]:
# Предсказание на тесте
predictions_lg_log = pipeline_lg.predict(X_test)
predictions_lg = np.expm1(predictions_lg_log)

In [12]:
# -----------------------------
# 5. Стекинг с кросс‑валидацией
# -----------------------------
print("Обучаем стекинг с кросс‑валидацией...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
stacking_features_train = []
y_oof_true = []
stacking_features_test = []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_vl = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr_log, y_vl_log = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]

    # Обучение базовых моделей
    pipeline_rf.fit(X_tr, y_tr_log)
    pipeline_xg.fit(X_tr, y_tr_log)
    pipeline_lg.fit(X_tr, y_tr_log)

    # OOF-предсказания
    pred_rf = pipeline_rf.predict(X_vl)
    pred_xg = pipeline_xg.predict(X_vl)
    pred_lg = pipeline_lg.predict(X_vl)
    stacking_features_train.append(np.column_stack([pred_rf, pred_xg, pred_lg]))
    y_oof_true.append(y_vl_log.values)

    # Предсказания на тесте
    pred_test_rf = pipeline_rf.predict(X_test)
    pred_test_xg = pipeline_xg.predict(X_test)
    pred_test_lg = pipeline_lg.predict(X_test)
    stacking_features_test.append(np.column_stack([pred_test_rf, pred_test_xg, pred_test_lg]))

# Подготовка данных для метамодели
X_meta = np.vstack(stacking_features_train)
y_meta = np.concatenate(y_oof_true)

scaler = StandardScaler()
X_meta_scaled = scaler.fit_transform(X_meta)

meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_scaled, y_meta)

# Прогноз на валидации
pred_rf_val = pipeline_rf.predict(X_val)
pred_xg_val = pipeline_xg.predict(X_val)
pred_lg_val = pipeline_lg.predict(X_val)
X_val_meta = np.column_stack([pred_rf_val, pred_xg_val, pred_lg_val])
X_val_meta_scaled = scaler.transform(X_val_meta)
y_pred_stacking_log = meta_model.predict(X_val_meta_scaled)
# Расчёт RMSLE для стекинга
rmsle_stacking = np.sqrt(mean_squared_log_error(
    np.expm1(y_val_log),
    np.expm1(y_pred_stacking_log)
))
print(f"RMSLE Stacking (CV): {rmsle_stacking:.4f}")

Обучаем стекинг с кросс‑валидацией...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2986
[LightGBM] [Info] Number of data points in the train set: 934, number of used features: 74
[LightGBM] [Info] Start training from score 12.030005
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001591 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3008
[LightGBM] [Info] Number of data points in the train set: 934, number of used features: 74
[LightGBM] [Info] Start training from score 12.034848
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001418 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2986
[LightGBM] [Info] Number of data points in the train set: 934, number of used featur

In [13]:
# -----------------------------
# 6. Формирование финальных сабмишенов
# -----------------------------
print("\nФормирование финальных сабмишенов...")

# Предсказания стекинга на тестовой выборке
X_test_meta = np.mean(np.array(stacking_features_test), axis=0)
X_test_meta_scaled = scaler.transform(X_test_meta)
final_predictions_stacking_log = meta_model.predict(X_test_meta_scaled)
final_predictions_stacking = np.expm1(final_predictions_stacking_log)

# Сабмишен RandomForest
submission_rf = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions_rf
})
submission_rf.to_csv('submission_random_forest.csv', index=False)
print("Сабмишен RandomForest сохранён как 'submission_random_forest.csv'")

# Сабмишен XGBoost
submission_xg = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions_xg
})
submission_xg.to_csv('submission_xgboost.csv', index=False)
print("Сабмишен XGBoost сохранён как 'submission_xgboost.csv'")


# Сабмишен LightGBM
submission_lg = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions_lg
})
submission_lg.to_csv('submission_lightgbm.csv', index=False)
print("Сабмишен LightGBM сохранён как 'submission_lightgbm.csv'")

# Сабмишен стекинга
submission_stacking = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions_stacking
})
submission_stacking.to_csv('submission_stacking.csv', index=False)
print("Сабмишен Stacking сохранён как 'submission_stacking.csv'")


Формирование финальных сабмишенов...
Сабмишен RandomForest сохранён как 'submission_random_forest.csv'
Сабмишен XGBoost сохранён как 'submission_xgboost.csv'
Сабмишен LightGBM сохранён как 'submission_lightgbm.csv'
Сабмишен Stacking сохранён как 'submission_stacking.csv'


In [14]:
# -----------------------------
# 7. Итоговый вывод метрик
# -----------------------------
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ")
print("="*40)
print(f"RandomForest:    {rmsle_rf:.4f}")
print(f"XGBoost:        {rmsle_xg:.4f}")
print(f"LightGBM:       {rmsle_lg:.4f}")
print(f"Stacking (CV):   {rmsle_stacking:.4f}")
print("="*40)

print("\nОписание подхода:")
print("- Предобработка: заполнение пропусков, Ordinal Encoding категориальных признаков, стандартизация числовых.")
print("- Целевая переменная: логарифмирована (log(1 + x)) для стабилизации обучения.")
print("- Модели-базовые: RandomForest, XGBoost, LightGBM с подобранными гиперпараметрами.")
print("- Стекинг: метамодель Ridge на OOF-предсказаниях базовых моделей (5 фолдов CV).")
print("- Оценка: RMSLE на валидационной выборке.")

ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ
RandomForest:    0.1450
XGBoost:        0.1370
LightGBM:       0.1362
Stacking (CV):   0.1379

Описание подхода:
- Предобработка: заполнение пропусков, Ordinal Encoding категориальных признаков, стандартизация числовых.
- Целевая переменная: логарифмирована (log(1 + x)) для стабилизации обучения.
- Модели-базовые: RandomForest, XGBoost, LightGBM с подобранными гиперпараметрами.
- Стекинг: метамодель Ridge на OOF-предсказаниях базовых моделей (5 фолдов CV).
- Оценка: RMSLE на валидационной выборке.
